In [5]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-5"

In [6]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    effort=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    # thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "adaptive",
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if effort:
        params["output_config"] = {"effort": effort}

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [7]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

In [11]:
with open("images/prop7.png", "rb") as f:
    image_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []

add_user_message(messages, [
    {
        "type": "image",
        "source": {
            "type": "base64",
            "media_type": "image/png",
            "data": image_bytes
        }
    },
    {
        "type": "text",
        "text": prompt
    }
])

chat(messages)

Message(id='msg_011CePxUMj5HSQzxXxbr5SGa', container=None, content=[ThinkingBlock(signature='EpcCCpABCBEYAipAz/j4qCl9qfHqFqJ8JmJFChJR/ICANvzAnQBOwkUV4xQs1Tyl2ZSnbmR4yZV3TXy4q+b9GCf4IZEwB6ONmBaG3zIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQwZGRiMTdjOS00N2FmLTQ3YzAtOWE0NS05OTAzYWRkNTc4MWOoAdrDt9QGEgwHawkVPHt0JjPNLVEaDCc/gmE6/wWGpe7DAyIw73QcR47Lo9TS/OxcUrMBmmQPsCW27KN0VH6IWa8pBqEvvJoT0HZNxjXmfns9Go54KjRRThWiqnxcWKhFYqMg9qbzw/SCXkCF0xb91BHM5Pg8lAYilcWN2huyv3hWlxv1LnvDTpB5GAE=', thinking='', type='thinking'), TextBlock(citations=None, text='1. **Residence identification:** The primary residence is a multi-winged structure with a light gray/tan roof located centrally in the image, surrounded almost entirely by dense forest canopy with no visible driveway or clear property boundaries in frame.\n\n2. **Tree overhang analysis:** Dense tree canopy directly overlaps the roofline on the south, west, and southeast edges of the structure, with dark shadow patterns on the roof indicating overhanging bra

In [14]:
with open("earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode('utf-8')

messages = []

add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes
        },
        "title": "earth.pdf",
        "citations": {"enabled": True}
    },
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans formed?"
    }
])

chat(messages)

Message(id='msg_011CePxxvvdNPKcsrkFuCWdq', container=None, content=[TextBlock(citations=None, text="Based on the document, Earth's atmosphere and oceans originated through volcanic processes early in the planet's history:\n\n", type='text'), TextBlock(citations=[CitationPageLocation(cited_text="[42]\r\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\r\n[43] Water vapor from\r\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\r\nand comets.\r\n", document_index=0, document_title='earth.pdf', end_page_number=5, file_id=None, start_page_number=4, type='page_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing. Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.", type='text'), TextBlock(citations=None, text="\n\nIt's worth noting that ", type='text'), TextBlock(citations=[CitationPageLocation(